# GAS-BayesSHAP — audit P1 items (unique-capped ShaplEIG, regimes ≥20/regime, hard high-dim games)

Closes the three open P1 items from the latest audit.  Orchestrates the real CLI scripts only — no duplicated algorithm.

## 0. Environment & config

In [ ]:
import sys, os, time, subprocess
from pathlib import Path
sys.path.insert(0, "..")
import gas_bayesshap

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
print("GAS-BayesSHAP", gas_bayesshap.__version__)

N_INST  = int(os.environ.get("N_INST", "10"))
CAPS    = os.environ.get("CAPS", "512,1024")
PER_REG = int(os.environ.get("PER_REG", "20"))
SKIP    = set(os.environ.get("GAS_SKIP", "").split(",")) - {""}

def run(*args, tag="", skip=False):
    if skip:
        print(f"--- SKIPPED: {tag or ' '.join(args)}"); return 0.0
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    dt = time.time() - t0
    if r.returncode != 0:
        err = (r.stderr or r.stdout or '').strip().splitlines()
        print(' | '.join(err[-6:]) if err else '<no output>')
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"N_INST={N_INST} CAPS={CAPS} PER_REG={PER_REG} SKIP={sorted(SKIP)}")

## A. Unique-query-capped GAS vs ShaplEIG (audit item 2)

`run_unique_capped_shaplEIG.py --n 10 --caps 512,1024` runs GAS (cache disabled → every draw is a unique evaluation; Stage-2 budget = cap − Stage-1) and the ShaplEIG port at the same caps, recording each method's ACTUAL unique evals.  **Honest caveat:** GAS's fixed Stage-1+init+pilot cost at M=11 is ~371 unique evals, so caps {64,128,256} cannot be honoured (`cap_honored=False`); {512,1024} are the meaningful sub-enumerative caps (both ≪ 2^11=2048).  **~1–1.5 h on a laptop** (60 ShaplEIG runs, each refits a GP per round).  Smoke: `N_INST=1 CAPS=512`.

In [ ]:
run("run_unique_capped_shaplEIG.py", "--n", str(N_INST), "--caps", CAPS,
    tag=f"A. unique-capped GAS vs ShaplEIG (N={N_INST}, caps={CAPS})", skip=False)

In [ ]:
import pandas as pd
p = ROOT / "main_results" / "paper_unique_capped_shaplEIG.csv"
if p.exists():
    d = pd.read_csv(p)
    print(d[["dataset", "instance", "unique_cap", "gas_rmse", "shaplEIG_rmse",
             "gas_unique_evals", "shaplEIG_unique_queries", "cap_honored",
             "unique_matched"]].to_string(index=False))
    hon = d[d.cap_honored]
    print("\n=== cap-honored summary ===")
    if not hon.empty:
        print(hon.groupby(["dataset", "unique_cap"]).agg(
            gas_rmse=("gas_rmse", "mean"),
            gas_unique=("gas_unique_evals", "mean"),
            shaplEIG_rmse=("shaplEIG_rmse", "mean"),
            shaplEIG_unique=("shaplEIG_unique_queries", "mean"),
        ).round(5).to_string())
    print(f"\ncap_honored: {int(hon.shape[0])}/{len(d)} rows")

## B. Regime semantics — ≥20 instances per named regime (item 3)

`regime_semantics.py --per-regime 20 --clusters 4` runs n = 80 instances (20 per cluster → 20 per named regime, incl. the suffixed clean-air subregimes).  **~15–25 min.**

In [ ]:
run("regime_semantics.py", "--per-regime", str(PER_REG), "--clusters", "4",
    "--eps", "0.05", "--budget", "3000",
    tag=f"B. regime semantics per-regime N={PER_REG}", skip="B" in SKIP)

In [ ]:
import pandas as pd
p = ROOT / "main_results" / "paper_regime_semantics_summary.csv"
if p.exists():
    d = pd.read_csv(p)
    print(d.to_string(index=False))
    print("\nmin per-regime N:", int(d["n"].min()), "(audit: >= 20)")

## C. Hard high-dim games: threshold + unanimity (item 4)

`probe_high_dim.py --game threshold|unanimity --budgets 50000,100000` at M=30, spec range.  Both games have closed-form exact Shapley values (validated vs brute force to machine precision at M≤12): threshold 2-of-4 (φ=0.0625 on 4 drivers), unanimity 4-way AND (φ=0.125 on 4 drivers).  Expect: GP control variate degrades vs the sparse game, the spec-range interval stays rigorous and certifies nothing falsely.  **~2 min each.**

In [ ]:
run("probe_high_dim.py", "--M", "30", "--budgets", "50000,100000",
    "--game", "threshold", "--mode", "spec",
    tag="C1. M=30 threshold game (spec)", skip="C" in SKIP)

In [ ]:
run("probe_high_dim.py", "--M", "30", "--budgets", "50000,100000",
    "--game", "unanimity", "--mode", "spec",
    tag="C2. M=30 unanimity game (spec)", skip="C" in SKIP)

In [ ]:
import pandas as pd
for game in ("threshold", "unanimity"):
    p = ROOT / "main_results" / f"paper_high_dim_M30_{game}_spec_summary.csv"
    if p.exists():
        d = pd.read_csv(p)
        print(f"\n=== {game} ===")
        print(d[["K", "status", "unique_coalition_evals", "unique_vs_2M_ratio",
                 "n_sign_certified", "rmse_vs_exact", "mean_width"]].to_string(index=False))
        print(f"  certificate_is_rigorous: {bool(d['certificate_is_rigorous'].all())} "
              f"(must be True: spec interval valid even under misspecification)")

## Expected runtime and honest notes
- **Full run ≈ 1.5–2.5 h** (A ~1–1.5 h, B ~15–25 min, C ~5 min).
- **Smoke:** `N_INST=1 CAPS=512 PER_REG=4 GAS_SKIP=C` (~5 min).
- A: rows with `cap_honored=False` (cap below GAS's ~371 fixed cost)
  are excluded from the summary; the CSV keeps them for honesty.
- Commit the resulting CSVs: `paper_unique_capped_shaplEIG.csv`,
  `paper_regime_semantics{,_summary}.csv`,
  `paper_high_dim_M30_{threshold,unanimity}_spec_summary.csv`.